## Notebook11

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
us_pop = (
    pl.read_csv(ub + "data/us_city_population.csv")
    .select(c.city, c.state, c.year, c.population)
    .filter(c.population.null_count().over(c.city) <= 1)
)

### Questions

Same table as last time: 289 American cities, one row per city per census from 1790
to 2010, population in thousands. A city that did not exist yet is recorded as `0`,
and since the counts are rounded to one decimal in thousands, a village of forty
people is also recorded as `0`. That is still true and it will still cost you
something.

Last week's windows were built from `.sum()` and `.mean()`, which do not care how
the rows are arranged, and from `.shift()`, which cares about nothing else. This
week's tools are all in the second family. Keep an eye on the sort.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. `.rank()` gives each value its position within the column, and `.over()` scopes
that position to a group. Rank the 2010 cities within their own state, then sort the
result by state and rank so you can read each state's list from the top.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .with_columns(rank = c.population.rank().over(c.state))
    .sort(c.state, c.rank)
)

**Answer**:


2. Turn it over with `descending=True`, then keep only the rank 1 rows to get the
largest city in each of the 46 states, sorted by population.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .with_columns(rank = c.population.rank(descending=True).over(c.state))
    .filter(c.rank == 1)
    .sort(c.population, descending=True)
)

Forty-six rows, from New York City at 8.2 million down to Billings, Montana at
104,170. Note that `.rank()` never looked at the order of the rows: it read the values
and worked out the positions itself. The `.sort()` at the end only decides what you
look at first. This is the opposite of `.shift()`, which has no opinion about values
and does nothing but count rows.

3. Now rank on the other axis. Instead of ranking cities within a state, rank them
within a census year, which gives every city its national position at every point in
its history. Follow Detroit down the column.

In [ ]:
(
    us_pop
    .with_columns(rank = c.population.rank(descending=True).over(c.year))
    .filter(c.city == "Detroit, MI")
    .sort(c.year)
    .select(c.year, c.population, c.rank)
)

Detroit is the 13th largest city in the table in 1900, the 4th largest by 1920, and
the 18th in 2010. Two things to notice. The ranks come back as floats rather than
integers, because tied values are given the average of the positions they span, and
the early ranks are enormous and slightly wrong for that reason: 73 cities are
recorded as `0` in 1900, they tie with each other, and all 73 of them get rank 253.
A rank computed over a column full of placeholder zeros ranks the placeholders too.

4. Ranks are more interesting as movement. Rank the cities within each year, keep
just 1900 and 2010, and then use `.shift()` from last week to subtract one rank from
the other. Sort so the biggest climbers come first.

In [ ]:
(
    us_pop
    .with_columns(rank = c.population.rank(descending=True).over(c.year))
    .filter(c.year.is_in([1900, 2010]))
    .sort(c.city, c.year)
    .with_columns(move = c.rank.shift(1).over(c.city) - c.rank)
    .filter(c.year == 2010)
    .sort(c.move, descending=True)
)

In [ ]:
(
    us_pop
    .with_columns(rank = c.population.rank(descending=True).over(c.year))
    .filter(c.year.is_in([1900, 2010]))
    .sort(c.city, c.year)
    .with_columns(move = c.rank.shift(1).over(c.city) - c.rank)
    .filter(c.year == 2010)
    .sort(c.move)
)

Virginia Beach climbed 214 places, followed by Aurora, Anchorage, Henderson, and
Mesa. The falls are just as orderly: Fall River, Lynn, and New Bedford are all
Massachusetts mill towns, and Duluth, Albany, and Cambridge are in among them. The
century moved the country to the Sun Belt and the numbers say so without being asked.
Note the order the two windows run in. The rank is computed over the whole table
first, so it means "position among all 289 cities that year," and only then does the
filter cut the table down to two years. Compute the rank after the filter and it
would mean "position among these two years," which is not a thing anyone wants.

5. `.cum_sum()` is a running total: each row adds itself to everything above it. Take
the 2010 census, sort the cities from largest to smallest, and compute the running
share of the 289-city total.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .sort(c.population, descending=True)
    .select(
        c.city,
        c.population,
        cum_share = c.population.cum_sum() / c.population.sum() * 100
    )
)

The ten largest cities are 29 percent of the total and it takes 40 of the 289 to pass
half. There are two aggregations in that one expression doing different jobs:
`.cum_sum()` changes on every row, and the plain `.sum()` next to it is a single fixed
number, the same on all 289 rows. Also note that the `.sort()` is not cosmetic here.
Sort the table differently and every number in the column changes, which was never
true of `.rank()`.

6. Add an `.over()` and the running total restarts in each group. Sort the cities by
state, and within state from largest to smallest, then accumulate within the state.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .sort(c.state, c.population, descending=[False, True])
    .with_columns(
        cum_share = c.population.cum_sum().over(c.state) / c.population.sum().over(c.state) * 100
    )
    .filter(c.state.is_in(["CA", "NY"]))
)

Both states are done in one screen and they could not look less alike. New York has
six cities in the table and the running total is essentially finished on its first
row: New York City is 90 percent of the state. California has 71 cities, starts with
Los Angeles at 21 percent, and needs ten of them to pass half. One state has a single
city with its own gravity and the other has a coastline of them, and the cumulative
column says so more clearly than either total does.

7. `.rolling_mean()` averages each row with its neighbors to smooth out the noise.
It is an ordered function like the others, so it needs a sort, and it needs an
`.over()` for exactly the same reason `.shift()` did. Leave the `.over()` off first
and look at what happens at a boundary.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(smooth = c.population.rolling_mean(window_size=3))
    .filter(c.city.is_in(["Akron, OH", "Albany, NY"]), c.year <= 1810)
)

**Answer**:


8. Do it properly. Compute each city's change since the previous census, as you did
last week, then smooth that change over a three-census window, scoping both to the
city. Read Detroit from 1900 on.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(change = c.population - c.population.shift(1).over(c.city))
    .with_columns(change_smooth = c.change.rolling_mean(window_size=3).over(c.city))
    .filter(c.city == "Detroit, MI", c.year >= 1900)
    .select(c.year, c.population, c.change, c.change_smooth)
)

The raw change column bounces: Detroit adds 575,000 people in the 1920s, then 55,000
in the 1930s, then 226,000 in the 1940s. Wars and depressions land on the decade they
land on. The smoothed column does what smoothing is for and shows one shape instead:
it peaks at the 1930 row, falls away across the next three censuses, turns negative at
the 1970 row, and never comes back. Smoothing does not add any information to the table.
It removes some, and the argument for doing it is that what it removes was noise, an
argument you have to make each time rather than assume.

9. Close with a picture. Give every city its share of the census-year total, then
keep six cities and draw their shares over time with `.geom_line()`.

In [ ]:
(
    us_pop
    .with_columns(share = c.population / c.population.sum().over(c.year) * 100)
    .filter(c.city.is_in([
        "New York City, NY", "Chicago, IL", "Detroit, MI",
        "Los Angeles, CA", "Phoenix, AZ", "Houston, TX"
    ]))
    .pipe(ggplot, aes(c.year, c.share, color="city"))
    .geom_line()
    .labs(x="Census year", y="Share of the 289 cities (%)", color="City")
)

The lines cross in the order the century did. Chicago peaks in 1900 at 9.5 percent and
Detroit in 1930 at 3.9, and both of them are falling by the time Los Angeles passes
Detroit in 1950 and Chicago in the 1980s. Houston and Phoenix spend a hundred years on
the floor of the chart and then climb. This figure is why `.over()` exists: it needs one point per city per
year, so an aggregated table could not have drawn it, because aggregation would have
collapsed the very rows the lines are made of.

Two more things about it. The `.with_columns()` comes before the `.filter()`, so the
denominator is all 289 cities and a share means a share of urban America. Filter first
and the denominator becomes the six cities you kept, New York starts the chart at 100
percent, and the plot answers a question nobody asked. And look at New York between
1890 and 1900, where the line jumps from 12 percent to 19. New York did not grow by
half in ten years. It annexed Brooklyn, which was a separate city until 1898 and is
not a separate row in this table. Every window function in this chapter did its
arithmetic on a definition of "New York City" that changed underneath it in 1898, and
no amount of `.over()` will tell you that. Only knowing something about the cities
will.